In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

#加载.env文件中的环境变量
load_dotenv()

#创建模型
model = init_chat_model("gpt-4o")

In [2]:
# 在记忆中存储一个用户信息
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
user_id="session_001"
story = "chitchat"
store.put((user_id, story), user_id, {"name": "John Smith", "preferences": "The user prefers you to write a brief and polite email."})

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel  
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

#@dataclass
class MyContext(BaseModel): #
    user_id:str
    user_name: str


@tool
def fetch_user_info(runtime: ToolRuntime[MyContext]) -> str:
    """通过 user_id 获取用户信息."""
    user_id = runtime.context.user_id  # 从Runtime中获取上下文，然后在上下文中获取 user_id

    # 从记忆中检索用户信息
    user_info = runtime.store.get((user_id, story), user_id)

    if user_info is None:
        raise ValueError(f"User with ID {user_id} not found.")
    return f"用户信息：{user_info}"

agent = create_agent(
    model="gpt-4o",
    tools=[fetch_user_info],
    context_schema=MyContext,
    store=store # 将 store 传递给 agent
)

In [4]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=MyContext(user_id="session_001", user_name="")
)

In [5]:
result

{'messages': [HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='ea781d3a-76cc-47cd-85f9-95af83ef0180'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 44, 'total_tokens': 55, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_4021733588', 'id': 'chatcmpl-EZ96Fs2aGo1VVaI3igb9Oq8mUaCya', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13e37-2bc7-77f3-b86e-1b790e23843d-0', tool_calls=[{'name': 'fetch_user_info', 'args': {}, 'id': 'call_LGi39oOk8EQGsquHtnu2cq1e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input